# MERG v2 — Stage 1 Reaction Detector (Y_reaction)

**Purpose:** Train the binary *reaction detector* for the **Macro Event Response Gate (MERG)**.
This is the first of the two MERG models and answers the simplest, most defensible question:

> *"Given the pre-event candlestick microstructure, will a high-impact economic release
> produce ANY directional move (up **or** down) in the minutes after it prints?"*

**Target (`Y`):** `targetSimple` binarised — see the dedicated *"Target variable"* section for
a clean explanation of what we are predicting.

**Why "Stage 1" and "leak-free":** a previous `MERG_v1` used all 15 M1 windows and reported
ROC-AUC ≈ 0.97. That number was inflated because windows `9 → 1` are **post-event** bars — a
model that sees the market's *reaction* is not predicting it, it is observing it. This notebook
trains on a **window prefix** (default `5` = the 5 pre-event bars only) so every input feature
exists *before* the release. That makes the model deployable in production.

**Pipeline overview**
1. Configuration (paths, window prefix, split dates, event filter)
2. Load + validate the exported macro dataset
3. Target variable explanation → build `y_reaction`
4. Feature engineering (candle anatomy + derived ratios, gated by `WINDOW_PREFIX`)
5. Event handling (frequency filter + one-hot top-N, categories fitted on TRAIN only)
6. Chronological split → train / val / test (test sealed)
7. Feature selection (noise-injection voting — same method as the EURUSD notebooks)
8. Model selection (purged nested CV + recency weights)
9. Final model (isotonic calibration + soft-vote ensemble)
10. Threshold calibration (F1-optimal on validation)
11. Sealed test evaluation (ROC-AUC / PR-AUC / calibration / per-event precision)
12. Save bundle to `models_bin/`


In [ ]:
# ── Core imports ────────────────────────────────────────────────────────────────
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python  : {sys.version}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")


## 1. Configuration

All paths and key parameters live in one place. Change `WINDOW_PREFIX`, the split dates, or the
event filter here — every downstream cell reads from these.


In [ ]:
from pathlib import Path

# ── Instrument & dataset ────────────────────────────────────────────────────────
PAIR      = "EURUSD"     # the exported dataset is EURUSD-only
TIMEFRAME = "M1"         # the 15 windows are 1-minute bars around each release

# ── Paths ───────────────────────────────────────────────────────────────────────
ROOT = Path("../..").resolve()            # root of ml-signal-service (two levels up)

RAW_FILE     = ROOT / "data" / "raw" / "macro" / "ExportedData.csv"
MODELS_DIR   = ROOT / "models_bin"
FEATURES_DIR = ROOT / "data" / "features"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Leak-free window prefix ─────────────────────────────────────────────────────
# The 15 windows are anchored to the release moment:
#
#     window 15..11  = 5 pre-event bars
#     window 10      = the release bar (the minute the news prints)
#     window 9..1    = 10 post-event bars  ← NOT usable for a forward prediction
#
# WINDOW_PREFIX = how many windows (counting back from 15) the model may see.
#     5  = pre-event bars only (true blind prediction)   ← default, leak-free
#     6  = + the release bar       7 = +1 post bar     10 = +5 post bars
#     15 = all bars (leaky — reproduces the inflated MERG_v1 result)
WINDOW_PREFIX = 5

# ── Event handling ──────────────────────────────────────────────────────────────
# Many of the 183 events appear only once or twice in 19 years — impossible to learn
# from. We (1) drop events that occur fewer than EVENT_MIN_COUNT times *in TRAIN*,
# then (2) one-hot the EVENT_TOP_K most frequent survivors and bucket the rest into
# "OTHER". Both decisions are fitted on TRAIN only to avoid peeking at val/test.
EVENT_MIN_COUNT = 10
EVENT_TOP_K     = 20

# ── Target mapping ──────────────────────────────────────────────────────────────
# Stage 1 binarises targetSimple: reaction (U or D) vs no-reaction (N).
REACTION_CLASSES = ("U", "D")   # these count as y=1; "N" is y=0

# ── Chronological split boundaries (Strategy A from the integration plan) ───────
# Time-based, no shuffling. Edit ONLY here.
TRAIN_END = "2023-01-01"   # train = everything strictly before this date
VAL_END   = "2024-01-01"   # val   = [TRAIN_END, VAL_END)
                           # test  = [VAL_END, present)  ← sealed, touched exactly once

# ── CV / training knobs ─────────────────────────────────────────────────────────
OUTER_FOLDS   = 5
INNER_FOLDS   = 4
PURGE_DAYS    = 30      # embargo between train/test within a fold
SEARCH_ITERS  = 60      # RandomSearchCV draws per inner CV
RECENCY_DECAY = 0.15    # sample weight halves roughly every ~4.6 years

# ── Gate operating threshold ──────────────────────────────────────────────────
# Stage 1 is a VETO, not a trade signal. The F1-optimal threshold (~0.28) over-fires
# (≈50% of events at <0.5 precision). The gate operates at a HIGH-confidence cutoff:
# at 0.60 the sealed-test precision is ~0.70 and signals fall to ~15/year. This value
# is written into the model bundle as `threshold` so the runtime gate reads it as its
# default (macro_event_responder.py loads bundle["threshold"]).
GATE_THRESHOLD = 0.60

print(f"Pair          : {PAIR}")
print(f"Raw file      : {RAW_FILE}  (exists={RAW_FILE.exists()})")
print(f"Window prefix : {WINDOW_PREFIX}  (5 = leak-free, 15 = leaky)")
print(f"Gate threshold: {GATE_THRESHOLD}  (saved as bundle default)")
print(f"Train         : < {TRAIN_END}")
print(f"Val           : {TRAIN_END} → {VAL_END}")
print(f"Test (sealed) : >= {VAL_END}")


## 2. Load Raw Data

One row per historical high-impact EURUSD release (2007 → 2026). Columns:

- `event`, `time` — metadata
- 45 feature columns `tWick_i / body_i / bWick_i` for `i = 15..1` — candle anatomy (already
  price-normalised, so the model learns *shape*, not absolute price levels)
- 4 target columns `target / targetSimple / target1 / target2`


In [ ]:
df = pd.read_csv(RAW_FILE)

# `time` is stored as "YYYY.MM.DD HH:MM" with NO timezone marker.
# ⚠ P0 caveat (flagged in the integration plan): broker-time vs UTC is unverified.
# We parse it verbatim and use it only as a monotonic chronological index; the split
# and CV rely on ordering, not on an absolute timezone. Resolve with the data owner
# before trusting a live gate.
df["time"] = pd.to_datetime(df["time"], format="%Y.%m.%d %H:%M")

df = df.sort_values("time").reset_index(drop=True)

print(f"Rows      : {len(df):,}")
print(f"Columns   : {list(df.columns)}")
print(f"Date range: {df['time'].min()}  →  {df['time'].max()}")


## 3. Validate the Dataset

Sanity checks before feature engineering:
- dtypes — `time` must be a proper timestamp; anatomy features numeric
- nulls — none expected
- duplicates — no repeated rows
- anatomy sanity — wicks must be non-negative (body is signed: +bullish / −bearish)
- target distribution — confirm the natural N / U / D imbalance


In [ ]:
# ── 1. dtypes ───────────────────────────────────────────────────────────────────
print("=== dtypes (metadata + a feature sample) ===")
print(df[["event", "time", "tWick15", "body15", "bWick15", "target", "targetSimple"]].dtypes)

# ── 2. Null check ───────────────────────────────────────────────────────────────
print(f"\nTotal nulls across dataset: {df.isnull().sum().sum()}")

# ── 3. Duplicate rows ───────────────────────────────────────────────────────────
print(f"Duplicate rows            : {df.duplicated().sum()}")

# ── 4. Anatomy sanity ───────────────────────────────────────────────────────────
# Wicks (tWick/bWick) are distances → must be >= 0. body is signed → can be negative.
wick_cols = [c for c in df.columns if c.startswith(("tWick", "bWick"))]
neg_wicks = int((df[wick_cols] < 0).sum().sum())
print(f"Negative wick values (should be 0): {neg_wicks}")

# ── 5. Target distribution ──────────────────────────────────────────────────────
print("\n=== targetSimple distribution ===")
print(df["targetSimple"].value_counts(normalize=True).round(4).to_string())

print("\n=== distinct labels per target column ===")
print(f"target        : {df['target'].nunique()} labels")
print(f"target1       : {df['target1'].nunique()} labels")
print(f"target2       : {df['target2'].nunique()} labels")
print(f"targetSimple  : {df['targetSimple'].nunique()} labels")


## 4. Target variable — `targetSimple` (the cleaned-up `Y`)

### What the dataset already gives us

Every row is one historical high-impact release. The dataset ships **four** target columns,
all encoding the *same* event at different granularities:

| Column | Classes | Meaning |
|---|---|---|
| `target`       | 186 | full post-event **sequence** plan (e.g. `U`, `2D`, `U|D`, `2D|U`) |
| `target1`      | 186 | a single operation but **with magnitude** (e.g. `2U` ≈ 20 pips up) |
| `target2`      | 186 | two-operation plan |
| `targetSimple` | **3** | the **net direction** of the post-event reaction: `U` / `D` / `N` |

We use **`targetSimple`** (per the dataset owner's guidance — it is the label that generalised
best): it collapses the rich 186-class sequence into the three outcomes that matter for a first
model — *move up*, *move down*, or *no meaningful move*.

### What U / D / N mean

- **`U` (up)** — price closed a net positive number of pips above the release moment.
- **`D` (down)** — price closed a net negative number of pips below the release moment.
- **`N` (neutral)** — the move stayed below the noise threshold (no tradeable reaction).

The class balance is **naturally imbalanced** — most releases do *not* move the market:

| Class | Approx. share |
|---|---|
| N (neutral) | ~62% |
| U (up)      | ~20% |
| D (down)    | ~19% |

### How we turn it into a binary label for Stage 1

This notebook is **Stage 1 — the Reaction Detector**. We deliberately do NOT predict the 3
classes directly (a 3-class model would lazily predict the 62% majority `N` and look better
than it is). Instead we ask the simpler question first:

$$\texttt{y\_reaction} = 1 \text{ if } \texttt{targetSimple} \in \{\text{U},\ \text{D}\},\quad
0 \text{ if } \texttt{targetSimple} = \text{N}$$

- `y_reaction = 1` → *"this event produces a directional move"*  (≈38% of rows)
- `y_reaction = 0` → *"this event does nothing"*                 (≈62% of rows)

A future **Stage 2** notebook answers *"which way?"* — a binary U-vs-D classifier trained only
on the `y_reaction == 1` rows.

### The leakage distinction (why this is a *clean* target)

- `targetSimple` is the **outcome** — it is derived from the post-event bars. Using it as the
  label is correct: it is exactly the thing we are trying to predict.
- The leakage in the old `MERG_v1` was **not** the target, it was the *features*: v1 fed
  windows `9 → 1` (post-event bars) into the model. Predicting an outcome from its own aftermath
  is cheating. Here we feed **only** the pre-event windows (see `WINDOW_PREFIX`), so the target
  stays honest.


In [ ]:
# Build the binary Stage-1 label from targetSimple (see "Target variable" section above).
df["y_reaction"] = df["targetSimple"].isin(REACTION_CLASSES).astype(int)

print("=== y_reaction (Stage-1 label) ===")
print(df["y_reaction"].value_counts(normalize=True).round(4).to_string())
print(f"\nPositive rate (reaction)              : {df['y_reaction'].mean()*100:.1f}%")
print(f"Majority baseline (always predict N)  : "
      f"{max(df['y_reaction'].mean(), 1 - df['y_reaction'].mean())*100:.1f}%")
print("→ a useful model must beat the majority baseline on PR-AUC / ROC-AUC.")


## 5. Feature Engineering

The raw data is *already* feature data (candle anatomy), so — unlike the EURUSD notebooks — there
is no OHLCV→indicator pipeline. Two things remain:

1. **Window-prefix gating** — keep only the `WINDOW_PREFIX` pre-event windows (leak-free).
2. **Derived shape ratios** — per-bar ratios that describe *shape* independent of scale, plus a
   few cross-window aggregates computed over the active (pre-event) windows only.


In [ ]:
# ── Window numbering reminder ───────────────────────────────────────────────────
#   15  14  13  12  11 | 10 |  9  8  7  6  5  4  3  2  1
#   ─── pre-event ──── | rel| ─── post-event (leaky) ───
# WINDOW_PREFIX selects the first `prefix` windows counting back from 15.

def active_windows(prefix):
    """Return the window indices the model may see, high→low.
    prefix=5 → [15,14,13,12,11]; prefix=6 → [... ,10]; prefix=15 → [15..1]."""
    return list(range(15, 15 - prefix, -1))


def build_features(data, prefix):
    """Build the leak-free feature matrix from raw candle anatomy.

    For every active window `i` we keep the raw (tWick, body, bWick) triple and derive
    three shape ratios that describe the bar independently of its absolute price level
    (the dataset is already price-normalised, but these make the model scale-robust):

        range_i      = tWick_i + |body_i| + bWick_i      (total candle length)
        body_ratio_i = |body_i| / range_i                (body vs wick share)
        wick_asym_i  = (tWick_i - bWick_i) / range_i     (upper vs lower wick bias)

    We then add a few cross-window aggregates (sum / mean / std) over the active windows.
    All transforms are row-wise or within-row aggregates over the pre-event window set,
    so nothing looks into the future.
    """
    df = data.copy()
    wins = active_windows(prefix)

    raw_cols, derived_cols = [], []
    for i in wins:
        t, b, bw = f"tWick{i}", f"body{i}", f"bWick{i}"
        raw_cols += [t, b, bw]

        rng = df[t] + df[b].abs() + df[bw]
        # guard flat bars (range==0) → define ratios as 0 instead of dividing by zero
        df[f"range_{i}"]      = rng
        df[f"body_ratio_{i}"] = (df[b].abs() / rng.replace(0, np.nan)).fillna(0)
        df[f"wick_asym_{i}"]  = ((df[t] - df[bw]) / rng.replace(0, np.nan)).fillna(0)
        derived_cols += [f"range_{i}", f"body_ratio_{i}", f"wick_asym_{i}"]

    # ── Cross-window aggregates (order-independent, pre-event only) ──────────────
    body_all  = df[[f"body{i}" for i in wins]]
    asym_all  = df[[f"wick_asym_{i}" for i in wins]]
    range_all = df[[f"range_{i}" for i in wins]]

    df["body_sum"]       = body_all.sum(axis=1)
    df["body_mean"]      = body_all.mean(axis=1)
    df["body_abs_sum"]   = body_all.abs().sum(axis=1)
    df["body_std"]       = body_all.std(axis=1).fillna(0)   # prefix≥2 avoids NaN; guard anyway
    df["wick_asym_mean"] = asym_all.mean(axis=1)
    df["range_sum"]      = range_all.sum(axis=1)
    df["range_mean"]     = range_all.mean(axis=1)

    feature_cols = raw_cols + derived_cols + [
        "body_sum", "body_mean", "body_abs_sum", "body_std",
        "wick_asym_mean", "range_sum", "range_mean",
    ]
    return df, feature_cols


df, FEATURE_COLS = build_features(df, WINDOW_PREFIX)

print(f"Active windows   : {active_windows(WINDOW_PREFIX)}")
print(f"Anatomy features : {len(FEATURE_COLS)}  (raw + derived + aggregates)")
print(f"Feature sample   : {FEATURE_COLS[:8]} ... {FEATURE_COLS[-7:]}")


## 6. Event Handling

Event identity is highly informative (rate decisions move price ~73% of the time; speeches
~12–22%). But 183 raw event names are too many and most are too rare to learn from. We:

1. **Frequency filter** — drop events with `< EVENT_MIN_COUNT` occurrences *in TRAIN*.
2. **One-hot top-K + OTHER** — the `EVENT_TOP_K` most frequent survivors get their own
   indicator column; the remaining frequent events are bucketed into `OTHER`.

Both steps are fitted on **TRAIN only**, then applied to val/test, so a live event maps to the
same encoding the model learned. (A production name-normalisation map is still required before
the gate fires live — see the integration plan.)


In [ ]:
# ── Fit event categories on TRAIN only ─────────────────────────────────────────
train_mask   = df["time"] < TRAIN_END
event_counts = df.loc[train_mask, "event"].value_counts()

frequent_events = event_counts[event_counts >= EVENT_MIN_COUNT].index.tolist()
top_k_events    = event_counts[event_counts >= EVENT_MIN_COUNT].head(EVENT_TOP_K).index.tolist()

# ── (1) Frequency filter — drop rows whose event is too rare to learn from ──────
n_before = len(df)
df = df[df["event"].isin(frequent_events)].reset_index(drop=True)
print(f"Frequency filter: dropped {n_before - len(df)} rows (rare events), kept {len(df)}")

# ── (2) One-hot top-K; the remaining *frequent* events → "OTHER" ─────────────────
def event_to_category(ev):
    return ev if ev in top_k_events else "OTHER"

df["event_cat"] = df["event"].map(event_to_category)
event_dummies   = pd.get_dummies(df["event_cat"], prefix="evt", dtype=int)
if "evt_OTHER" not in event_dummies.columns:      # ensure OTHER always exists
    event_dummies["evt_OTHER"] = 0

# Attach the event columns to df so they travel with the train/val/test splits.
df = pd.concat([df, event_dummies], axis=1)
EVENT_COLS = list(event_dummies.columns)

print(f"Events kept                 : {len(frequent_events)}")
print(f"One-hot top-{EVENT_TOP_K} + OTHER : {len(EVENT_COLS)} columns")
print(f"\nEvent columns ({len(EVENT_COLS)}):")
print(EVENT_COLS)


In [ ]:
# ── Speech family flag ────────────────────────────────────────────────────────
# Speeches ("... speech", "... testifies") are a distinct regime: much lower reaction
# base rate (~13% vs ~35%) and the model over-predicts reactions on them when it lacks
# the "this is a speech" prior (the per-event one-hot for Lagarde et al. did not survive
# feature selection). A single dense binary flag is cheap, covers every speech event, and
# lets the noise-vote selector decide whether it carries real signal.
df["is_speech"] = (
    df["event"].str.lower().str.contains("speech|testifies", regex=True).astype(int)
)

FEATURE_COLS = FEATURE_COLS + ["is_speech"]

print(f"Speech rows    : {int(df['is_speech'].sum())} / {len(df)}")
print(f"FEATURE_COLS now includes `is_speech` ({len(FEATURE_COLS)} anatomy + flag).")


## 7. Chronological Split — Train / Validation / Test

Time-based split with **no shuffling** (same discipline as the EURUSD notebooks). The label was
assigned *before* splitting — `targetSimple` is already the outcome, so there is no leakage in
labelling before splitting.

| Set | Period | Purpose |
|---|---|---|
| **Train** | 2007 → 2022 | model fitting |
| **Validation** | 2023 | model selection + threshold calibration |
| **Test** | 2024 → present | sealed hold-out — opened once at the very end |

> Split dates live in the **Configuration** cell — change them there, not here.


In [ ]:
# ── Assign each row to a split by its release time ─────────────────────────────
def assign_split(t):
    if t < pd.Timestamp(TRAIN_END):
        return "train"
    elif t < pd.Timestamp(VAL_END):
        return "val"
    else:
        return "test"

df["split"] = df["time"].map(assign_split)

df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val   = df[df["split"] == "val"].reset_index(drop=True)
df_test  = df[df["split"] == "test"].reset_index(drop=True)

# ── Class balance check ──────────────────────────────────────────────────────────
print("=== Class balance (y_reaction) ===")
print(f"{'Split':<7}{'Total':>8}{'react=1':>10}{'neutral=0':>11}{'%react':>9}")
print("-" * 45)
for name, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    n = len(d); pos = int(d["y_reaction"].sum())
    print(f"{name:<7}{n:>8,}{pos:>10,}{n - pos:>11,}{pos / n * 100:>8.1f}%")


## 8. Feature Selection

**Method — noise features + voting system** (identical to the EURUSD notebooks).

We inject random *noise* features alongside the real features, train three models, and read
their importances. Any real feature that scores **below the noise** is dropped. A feature is
kept only if it consistently beats the noise across the strategies.

> Everything here is fit on **`df_train` only**. Validation and test are never touched during
> feature selection — otherwise we would leak future information.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

# ── Assemble final feature matrix = anatomy features + event one-hot ────────────
# (the event columns were already attached to df, so each split carries them)
def make_matrix(d):
    return d[FEATURE_COLS + EVENT_COLS].copy()

X_train = make_matrix(df_train).fillna(0)
y_train = df_train["y_reaction"].astype(int)

print(f"Real features : {X_train.shape[1]}  ({len(FEATURE_COLS)} anatomy + {len(EVENT_COLS)} event)")
print(f"X_train shape : {X_train.shape}")
print(f"Positive rate : {y_train.mean()*100:.1f}%")

# ── Generate noise features (fit-free, seeded for reproducibility) ──────────────
np.random.seed(42)
n = len(X_train)

noise = {
    "noise_gaussian_1":  np.random.normal(0, 1, n),
    "noise_gaussian_2":  np.random.normal(0, 2, n),
    "noise_gaussian_3":  np.random.normal(5, 1, n),
    "noise_uniform_1":   np.random.uniform(-1, 1, n),
    "noise_uniform_2":   np.random.uniform(-10, 10, n),
    "noise_poisson_1":   np.random.poisson(3, n),
    "noise_poisson_2":   np.random.poisson(6, n),
    "noise_random_walk": np.cumsum(np.random.normal(0, 1, n)),
    "noise_sinusoidal":  np.sin(np.linspace(0, 10, n)) + np.random.normal(0, 0.1, n),
}
noise_features = list(noise.keys())

X_noise = X_train.copy()
for name, values in noise.items():
    X_noise[name] = values

print(f"Noise features added : {len(noise_features)}")
print(f"X_noise shape        : {X_noise.shape}")


In [ ]:
# ── Train three models to extract feature importances ─────────────────────────
# class_weight="balanced" handles the ~38/62 imbalance without resampling.
#
# ⚠ IMPORTANCE-MEASUREMENT FIX — why we do NOT use raw tree importances here:
#   LightGBM's default `feature_importances_` is a raw *split count* on a very
#   different scale (thousands) than RF (0–1) or LogReg |coef| (~0.05). A weighted
#   average of raw scales collapses into "LightGBM split count", and deep trees split
#   *smooth continuous noise* (random walk, sinusoid) many times — so noise topped the
#   ranking. To fix it we:
#     1. train on the first 75% of train (chronological) and hold out the last 25%
#        for a *permutation importance* score — a held-out score cannot be gamed by an
#        overfit model memorising noise;
#     2. use permutation importance for the trees and |coef| for the linear model;
#     3. min-max-normalise each model's importance to [0,1] so the three are comparable.

from sklearn.inspection import permutation_importance

# Hold out the last 25% of train (chronologically) for importance scoring.
n_fit = int(len(X_noise) * 0.75)
X_fit, X_perm = X_noise.iloc[:n_fit], X_noise.iloc[n_fit:]
y_fit, y_perm = y_train.iloc[:n_fit], y_train.iloc[n_fit:]

# 1) Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_split=10, min_samples_leaf=5,
    max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1,
)
rf.fit(X_fit, y_fit)

# 2) LightGBM
print("Training LightGBM...")
lgbm = lgb.LGBMClassifier(
    n_estimators=500, max_depth=16, learning_rate=0.05, num_leaves=150,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, verbose=-1,
)
lgbm.fit(X_fit, y_fit)

# 3) Logistic Regression (linear) — needs scaling; |coef| is its importance.
print("Training Logistic Regression...")
scaler = StandardScaler()
X_fit_scaled = pd.DataFrame(scaler.fit_transform(X_fit.fillna(0)), columns=X_fit.columns)

logreg = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=42)
logreg.fit(X_fit_scaled, y_fit)

# ── Collect importances (all on a comparable footing) ─────────────────────────
# Permutation importance = mean drop in PR-AUC when a feature is shuffled, measured
# on the held-out 25% of train. Shuffling true noise → ~0 (or negative) importance.
rf_imp = permutation_importance(
    rf, X_perm, y_perm, scoring="average_precision",
    n_repeats=10, random_state=42, n_jobs=-1,
).importances_mean
lgbm_imp = permutation_importance(
    lgbm, X_perm, y_perm, scoring="average_precision",
    n_repeats=10, random_state=42, n_jobs=-1,
).importances_mean
logreg_imp = np.abs(logreg.coef_).ravel()   # |coefficient| = linear importance


def norm01(v):
    """Min-max normalise an importance vector to [0,1] (0 when constant)."""
    lo, hi = v.min(), v.max()
    return (v - lo) / (hi - lo) if hi > lo else np.zeros_like(v)


rf_imp     = norm01(rf_imp)
lgbm_imp   = norm01(lgbm_imp)
logreg_imp = norm01(logreg_imp)

print("✓ All 3 models trained; importances normalised to [0,1]")


In [ ]:
# ── Build the importance table ────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "feature":    X_noise.columns,
    "rf_imp":     rf_imp,
    "lgbm_imp":   lgbm_imp,
    "logreg_imp": logreg_imp,
})
imp_df["is_noise"] = imp_df["feature"].isin(noise_features)

# Equal-weight average of the three NORMALISED importances (all on [0,1]).
imp_df["avg_imp"] = (imp_df["rf_imp"] + imp_df["lgbm_imp"] + imp_df["logreg_imp"]) / 3
imp_df = imp_df.sort_values("avg_imp", ascending=False).reset_index(drop=True)

# ── Voting system ─────────────────────────────────────────────────────────────
# Each model "votes" for a feature if its importance is above the given percentile.
VOTING_PERCENTILE = 45     # lower = more inclusive, higher = stricter

rf_thr     = np.percentile(rf_imp, VOTING_PERCENTILE)
lgbm_thr   = np.percentile(lgbm_imp, VOTING_PERCENTILE)
logreg_thr = np.percentile(logreg_imp, VOTING_PERCENTILE)

imp_df["rf_vote"]     = (imp_df["rf_imp"]     >= rf_thr).astype(int)
imp_df["lgbm_vote"]   = (imp_df["lgbm_imp"]   >= lgbm_thr).astype(int)
imp_df["logreg_vote"] = (imp_df["logreg_imp"] >= logreg_thr).astype(int)
imp_df["total_votes"] = imp_df[["rf_vote", "lgbm_vote", "logreg_vote"]].sum(axis=1)

print(f"Voting thresholds (top {VOTING_PERCENTILE}%):  "
      f"RF={rf_thr:.4f}  LGBM={lgbm_thr:.4f}  LogReg={logreg_thr:.4f}\n")
print("Top 10 features by average importance:")
print(imp_df[["feature", "avg_imp", "total_votes", "is_noise"]].head(10).to_string(index=False))


In [ ]:
# ── Noise benchmark ───────────────────────────────────────────────────────────
noise_df = imp_df[imp_df["is_noise"]]
real_df  = imp_df[~imp_df["is_noise"]]

noise_imp_mean  = noise_df["avg_imp"].mean()
noise_imp_std   = noise_df["avg_imp"].std()
noise_imp_p70   = np.percentile(noise_df["avg_imp"], 70)
noise_votes_max = noise_df["total_votes"].max()

print(f"Noise benchmark — mean:{noise_imp_mean:.5f}  p70:{noise_imp_p70:.5f}  "
      f"max_votes:{noise_votes_max}\n")

# ── Five selection strategies (each returns a set of real features) ───────────
best_noise_rank = imp_df[imp_df["is_noise"]].index.min()   # rank of the best noise feature

strategies = {
    "better_than_best_noise": set(
        real_df[real_df.index < best_noise_rank]["feature"]
    ),
    "above_noise_p70": set(
        real_df[real_df["avg_imp"] > noise_imp_p70]["feature"]
    ),
    "more_votes_than_noise": set(
        real_df[real_df["total_votes"] > noise_votes_max]["feature"]
    ),
    "statistical_threshold": set(
        real_df[real_df["avg_imp"] > noise_imp_mean + 0.5 * noise_imp_std]["feature"]
    ),
    "vote_and_above_mean": set(
        real_df[(real_df["total_votes"] >= 1) & (real_df["avg_imp"] > noise_imp_mean)]["feature"]
    ),
}

for name, feats in strategies.items():
    print(f"  {name:<25} {len(feats)} features")

# ── Consensus — count how many strategies support each feature ────────────────
support = {}
for feats in strategies.values():
    for f in feats:
        support[f] = support.get(f, 0) + 1

MIN_STRATEGY_SUPPORT = 2   # keep features backed by at least this many strategies
selected_features = sorted(
    [f for f, s in support.items() if s >= MIN_STRATEGY_SUPPORT],
    key=lambda f: imp_df.loc[imp_df["feature"] == f, "avg_imp"].iloc[0],
    reverse=True,
)

# Safety net: make sure noise never slips through
selected_features = [f for f in selected_features if f not in noise_features]

print(f"\n✓ Selected features (≥{MIN_STRATEGY_SUPPORT} strategies): {len(selected_features)}")

# ── Guard: if NOTHING beats noise, that is itself the leakage verdict ──────────
if len(selected_features) == 0:
    print("\n⚠ NO real feature beat the noise — the pre-event candle anatomy + event")
    print("  identity carry no detectable forward signal at this window prefix.")
    print("  (If this persists at prefix=15 too, MERG has no predictive value.)")
    print("  Falling back to ALL real features so the pipeline still completes and you")
    print("  can read the (expected ≈random) evaluation numbers below.")
    selected_features = FEATURE_COLS + EVENT_COLS


In [ ]:
# ── Report ────────────────────────────────────────────────────────────────────
print("=" * 60)
print("FEATURE SELECTION RESULTS")
print("=" * 60)
print(f"Real features tested : {len(FEATURE_COLS) + len(EVENT_COLS)}")
print(f"Selected             : {len(selected_features)}")
print(f"Reduction            : {(1 - len(selected_features) / len(X_noise.columns)) * 100:.0f}%")
print(f"Best noise importance: {noise_df['avg_imp'].max():.5f}")

selected_report = imp_df[imp_df["feature"].isin(selected_features)]
print("\nTop 15 selected features:")
print(selected_report[["feature", "avg_imp", "total_votes"]].head(15).to_string(index=False))

# Which event columns (if any) survived? This tells us whether event identity carries
# signal beyond the candle anatomy.
event_survivors = [f for f in selected_features if f.startswith("evt_")]
print(f"\nEvent features that survived selection: {len(event_survivors)}")
if event_survivors:
    print(event_survivors)

# ── Visualization ─────────────────────────────────────────────────────────────
top = imp_df.head(40)
colors = [
    "green" if f in selected_features else ("red" if n else "lightgray")
    for f, n in zip(top["feature"], top["is_noise"])
]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(16, 6))

axL.bar(range(len(top)), top["avg_imp"], color=colors)
axL.axhline(noise_df["avg_imp"].max(), color="red", linestyle="--",
            label=f"best noise ({noise_df['avg_imp'].max():.4f})")
axL.axhline(noise_imp_mean, color="orange", linestyle="--",
            label=f"noise mean ({noise_imp_mean:.4f})")
axL.set_xticks(range(len(top)))
axL.set_xticklabels(top["feature"], rotation=90, fontsize=7)
axL.set_ylabel("avg importance")
axL.set_title(f"MERG {PAIR} {TIMEFRAME} — importance (top 40)  [green=kept, red=noise]")
axL.legend()

axR.bar(range(len(top)), top["total_votes"], color=colors)
axR.axhline(noise_votes_max, color="red", linestyle="--",
            label=f"best noise votes ({noise_votes_max})")
axR.set_xticks(range(len(top)))
axR.set_xticklabels(top["feature"], rotation=90, fontsize=7)
axR.set_ylabel("total votes")
axR.set_title("voting pattern (top 40)")
axR.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── Persist the selection for the training phase ──────────────────────────────
selected_report.to_csv(FEATURES_DIR / f"{PAIR}_MERG_v2_stage1_selected_features.csv", index=False)
imp_df.to_csv(FEATURES_DIR / f"{PAIR}_MERG_v2_stage1_feature_importance_full.csv", index=False)

print(f"✓ Saved {len(selected_features)} selected features to:")
print(f"  {FEATURES_DIR / f'{PAIR}_MERG_v2_stage1_selected_features.csv'}")
print(f"\nselected_features ready for training ({len(selected_features)} columns).")


### 8.2 Model Selection

We pick the best classifier with **nested cross-validation** — an unbiased way to compare models
that also tunes hyperparameters without leaking val/test data.

- **Outer CV** — purged expanding `TimeSeriesSplit` (train on past, test on future) with a
  `PURGE_DAYS` embargo so neighbouring events don't straddle a fold boundary.
- **Inner CV** — `TimeSeriesSplit` for hyperparameter search.
- **Recency weights** — recent years get more influence (markets drift).
- **Optimization metric** — **PR-AUC** (average precision), the right target for this imbalanced
  ~38/62 reaction problem.

Progression: **Logistic Regression** (baseline) → **Random Forest** → **XGBoost** → **LightGBM**.


In [ ]:
# ── Build model matrices from the selected features ───────────────────────────
X_train = df_train[selected_features].fillna(0)
y_train = df_train["y_reaction"].astype(int)

X_val = df_val[selected_features].fillna(0)
y_val = df_val["y_reaction"].astype(int)

X_test = df_test[selected_features].fillna(0)   # sealed — not touched until the very end
y_test = df_test["y_reaction"].astype(int)

# ── Purged expanding TimeSeriesSplit ───────────────────────────────────────────
# Expanding windows with a PURGE_DAYS embargo between train and test within each fold.
# Tests the skill that matters in production: train on the past, predict the future.
# Assumes df_train is already sorted chronologically.
train_dates = pd.to_datetime(df_train["time"])
n_train = len(df_train)

outer_cv, fold_boundaries = [], []
block_size = n_train // (OUTER_FOLDS + 1)

for fold in range(OUTER_FOLDS):
    test_start_raw = block_size * (fold + 1)
    test_end_raw   = block_size * (fold + 2) if fold < OUTER_FOLDS - 1 else n_train

    # Purge: slide test_start forward until >= PURGE_DAYS after the last train bar
    last_train_date = train_dates.iloc[test_start_raw - 1]
    purge_threshold = last_train_date + pd.Timedelta(days=PURGE_DAYS)

    test_start = test_start_raw
    while test_start < test_end_raw and train_dates.iloc[test_start] < purge_threshold:
        test_start += 1

    if test_end_raw - test_start < 50:
        continue   # fold too small to evaluate — skip

    outer_cv.append((np.arange(0, test_start), np.arange(test_start, test_end_raw)))
    fold_boundaries.append((train_dates.iloc[test_start], train_dates.iloc[test_end_raw - 1]))

# ── Recency sample weights ────────────────────────────────────────────────────
# Recent years matter more — weight decays ~15% per year back in time.
train_years = df_train["time"].dt.year.values

def time_decay_weights(years, decay=RECENCY_DECAY):
    years_ago = years.max() - years
    w = np.exp(-decay * years_ago)
    return w * len(w) / w.sum()   # normalise so weights sum to n

sample_weights = time_decay_weights(train_years)

print(f"Features : {len(selected_features)}")
print(f"Train    : {X_train.shape}  (pos {y_train.mean()*100:.1f}%)")
print(f"Val      : {X_val.shape}  (pos {y_val.mean()*100:.1f}%)")
print(f"Test     : {X_test.shape}  (sealed)")
print(f"Years    : {sorted(set(train_years))}")
print(f"Outer CV : purged expanding TimeSeriesSplit ({len(outer_cv)} folds, {PURGE_DAYS}-day embargo)")
for i, (s, e) in enumerate(fold_boundaries):
    print(f"  fold {i+1} test: {s.strftime('%Y-%m-%d')} → {e.strftime('%Y-%m-%d')}")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from xgboost import XGBClassifier

# scale_pos_weight balances XGBoost the way class_weight="balanced" does for the others
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "LogReg": {
        "model": Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
        ]),
        "params": {},   # baseline — no tuning
    },
    "RandomForest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        "params": {
            "n_estimators": [200, 400, 600],
            "max_depth": [10, 15, 20, None],
            "min_samples_leaf": [5, 10, 20, 50],
            "max_features": ["sqrt", "log2", 0.5],
        },
    },
    "XGBoost": {
        "model": XGBClassifier(
            random_state=42, n_jobs=-1, tree_method="hist",
            eval_metric="logloss", scale_pos_weight=pos_weight,
        ),
        "params": {
            "max_depth": [5, 7, 9, 11],
            "learning_rate": [0.05, 0.10, 0.20],
            "n_estimators": [150, 250, 400],
            "subsample": [0.7, 0.8, 0.9],
            "colsample_bytree": [0.7, 0.8, 0.9],
            "reg_lambda": [0.5, 1.0, 5.0],
        },
    },
    "LightGBM": {
        "model": lgb.LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1),
        "params": {
            "max_depth": [6, 8, 10, 12],
            "num_leaves": [15, 31, 63, 127],
            "learning_rate": [0.05, 0.10, 0.20],
            "n_estimators": [150, 250, 400],
            "subsample": [0.8, 0.9, 1.0],
            "colsample_bytree": [0.7, 0.8, 0.9],
            "reg_lambda": [0.1, 0.5, 1.0, 5.0],
            "min_child_samples": [10, 20, 50],
        },
    },
}

INNER_CV = TimeSeriesSplit(n_splits=INNER_FOLDS)

print(f"Inner CV : TimeSeriesSplit({INNER_FOLDS})")
print(f"Search   : {SEARCH_ITERS} iters | optimizing PR-AUC (average_precision)")
print(f"XGB scale_pos_weight = {pos_weight:.2f}")


In [ ]:
from sklearn.metrics import average_precision_score


def nested_cv(model, params, X, y, weights, name):
    """Nested cross-validation for one classifier.

    Inner loop tunes hyperparameters (RandomizedSearchCV, PR-AUC).
    Outer loop uses purged expanding TimeSeriesSplit — train on past, test on future.
    Returns scores + best params per fold.
    """
    outer_scores, best_params_per_fold = [], []

    for fold, (tr, te) in enumerate(outer_cv, 1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        w_tr = weights[tr]

        if params:  # tree models — tune hyperparameters on the inner split
            search = RandomizedSearchCV(
                model, params, n_iter=SEARCH_ITERS, cv=INNER_CV,
                scoring="average_precision", n_jobs=-1, random_state=42,
            )
            search.fit(X_tr, y_tr, sample_weight=w_tr)
            best = search.best_estimator_
            best_params_per_fold.append(search.best_params_)
        else:       # baseline — no tuning
            best = model.fit(X_tr, y_tr)
            best_params_per_fold.append({})

        # PR-AUC on the untouched outer fold
        proba = best.predict_proba(X_te)[:, 1]
        outer_scores.append(average_precision_score(y_te, proba))
        print(f"   fold {fold}/{len(outer_cv)}  PR-AUC = {outer_scores[-1]:.3f}")

    print(f"   → {name}: PR-AUC {np.mean(outer_scores):.3f} ± {np.std(outer_scores):.3f}")
    return {
        "name": name,
        "pr_auc_mean": np.mean(outer_scores),
        "pr_auc_std": np.std(outer_scores),
        "best_params_per_fold": best_params_per_fold,
    }


print("✓ nested_cv() ready")


In [ ]:
# ── Run nested CV for every model ─────────────────────────────────────────────
cv_results = {}
for name, cfg in models.items():
    print(f"\n{name}")
    cv_results[name] = nested_cv(
        cfg["model"], cfg["params"], X_train, y_train, sample_weights, name,
    )

# ── Comparison table (higher PR-AUC is better) ────────────────────────────────
comparison = (
    pd.DataFrame([
        {"model": r["name"], "pr_auc": r["pr_auc_mean"], "std": r["pr_auc_std"]}
        for r in cv_results.values()
    ])
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

# Baseline reference = positive rate (a random model scores ≈ prevalence)
baseline_pr = y_train.mean()

print("\n=== Model comparison (PR-AUC) ===")
print(comparison.round(3).to_string(index=False))
print(f"\nRandom baseline PR-AUC ≈ {baseline_pr:.3f} (positive rate)")

best_model_name = comparison.iloc[0]["model"]
print(f"\n🏆 Best model: {best_model_name}")


In [ ]:
# ── Manual model override ─────────────────────────────────────────────────────
# By default the best-CV model is used. To force a different one, set MANUAL_MODEL to
# "LogReg", "RandomForest", "XGBoost" or "LightGBM" and re-run this cell + everything after.
MANUAL_MODEL = None

if MANUAL_MODEL is not None:
    if MANUAL_MODEL not in cv_results:
        raise ValueError(f"Unknown model '{MANUAL_MODEL}'. Choose from: {list(cv_results.keys())}")
    best_model_name = MANUAL_MODEL
    print(f"⚠ Manual override active → using {best_model_name}")
else:
    print(f"✓ Auto-selected best model → {best_model_name}")

chosen = cv_results[best_model_name]
print(f"\n  CV PR-AUC : {chosen['pr_auc_mean']:.3f} ± {chosen['pr_auc_std']:.3f}")
print(f"\nModel comparison for reference:")
print(comparison.round(3).to_string(index=False))


In [ ]:
# ── Final model: isotonic calibration + optional soft-vote ensemble ──────────
from collections import Counter
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix,
)

USE_CALIBRATION = True    # isotonic calibration → reliable, monotonic probabilities
USE_ENSEMBLE    = True    # average all four calibrated models instead of one winner
CALIB_CV        = 3       # internal CV folds for the calibrator (fit on TRAIN only)


def most_frequent_params(params_list):
    """Aggregate the hyperparameters most often chosen across the outer folds."""
    final = {}
    for key in {k for p in params_list for k in p}:
        vals = [p[key] for p in params_list if key in p]
        final[key] = Counter(vals).most_common(1)[0][0]
    return final


def build_tuned(name):
    """Fresh estimator with its most-frequent CV hyperparameters applied."""
    est = models[name]["model"]
    params = most_frequent_params(cv_results[name]["best_params_per_fold"])
    if params:
        est.set_params(**params)
    return est, params


def make_calibrated(est):
    """Wrap an estimator in isotonic calibration (internal CV on train)."""
    return CalibratedClassifierCV(est, method="isotonic", cv=CALIB_CV) if USE_CALIBRATION else est


# ── Assemble the final estimator ───────────────────────────────────────────────
if USE_ENSEMBLE:
    estimators = []
    for name in models:
        base, params = build_tuned(name)
        estimators.append((name, make_calibrated(base)))
        print(f"  {name:<12} params: {params if params else '(baseline)'}")
    final_model = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
    final_model.fit(X_train, y_train)          # ensemble of pipelines → fit without sample_weight
    best_model_name = "Ensemble"
    print("\n✓ Final model = soft-vote ensemble of 4 calibrated models")
else:
    base, params = build_tuned(best_model_name)
    final_model = make_calibrated(base)
    try:
        final_model.fit(X_train, y_train, sample_weight=sample_weights)
    except TypeError:
        final_model.fit(X_train, y_train)
    print(f"Final hyperparameters: {params}")
    print(f"\n✓ Final model = {best_model_name}{' (calibrated)' if USE_CALIBRATION else ''}")

# ── Validation performance (0.5 is a reference only — threshold tuned next) ────
val_proba = final_model.predict_proba(X_val)[:, 1]
val_pred  = (val_proba >= 0.5).astype(int)

print(f"\n=== Validation performance (@0.5 reference) — {best_model_name} ===")
print(f"Precision : {precision_score(y_val, val_pred, zero_division=0):.3f}")
print(f"Recall    : {recall_score(y_val, val_pred, zero_division=0):.3f}")
print(f"F1        : {f1_score(y_val, val_pred, zero_division=0):.3f}")
print(f"ROC-AUC   : {roc_auc_score(y_val, val_proba):.3f}")
print(f"PR-AUC    : {average_precision_score(y_val, val_proba):.3f}")
print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_val, val_pred))


### 8.3 Threshold Calibration

Unlike the EURUSD models there is **no trading breakeven** here — Stage 1 is a *reaction
detector*, not a trade signal. We pick the threshold that maximises **F1** on validation,
subject to a minimum-support floor (so the choice isn't driven by a handful of lucky rows).

> In production the gate applies its own confidence threshold (`confidence_threshold`, default
> 0.60) on top of this operating point, so this threshold is a sane default, not a hard gate.


In [ ]:
from sklearn.metrics import precision_recall_curve

# ── F1-optimal threshold with a minimum-support floor ──────────────────────────
MIN_VAL_SIGNALS = 50   # ignore thresholds that fire on too few validation rows

precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)

rows = []
for thr, p, r in zip(thresholds, precisions[:-1], recalls[:-1]):
    n_sig = int((val_proba >= thr).sum())
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    rows.append({"threshold": round(float(thr), 4), "precision": round(float(p), 4),
                 "recall": round(float(r), 4), "f1": round(float(f1), 4),
                 "n_signals": n_sig})
cal_df = pd.DataFrame(rows)

eligible = cal_df[cal_df["n_signals"] >= MIN_VAL_SIGNALS]
if not eligible.empty:
    best = eligible.sort_values("f1", ascending=False).iloc[0]
    OPTIMAL_THRESHOLD = float(best["threshold"])
    print(f"Min signals floor : {MIN_VAL_SIGNALS}")
    print(f"Optimal threshold : {OPTIMAL_THRESHOLD}")
    print(f"Precision @ opt   : {best['precision']}")
    print(f"Recall    @ opt   : {best['recall']}")
    print(f"F1        @ opt   : {best['f1']}")
    print(f"Signals           : {int(best['n_signals'])}")
    print("\nTop eligible thresholds (by F1):")
    print(eligible.sort_values("f1", ascending=False)
                .drop(columns="f1").head(10).to_string(index=False))
else:
    OPTIMAL_THRESHOLD = 0.5
    print(f"⚠ No threshold reaches {MIN_VAL_SIGNALS} signals — falling back to 0.5.")

# ── Precision-recall curve ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(recalls[:-1], precisions[:-1], color="steelblue", lw=1.5, label="PR curve")
ax.axhline(y_val.mean(), color="gray", ls=":", lw=1, label=f"Base rate {y_val.mean():.2f}")
ax.axvline(best["recall"], color="green", ls="--", lw=1, label=f"Optimal recall {best['recall']:.2f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"MERG {PAIR} {TIMEFRAME} — Precision-Recall (validation)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ── Note: which threshold do we actually ship? ─────────────────────────────────
# OPTIMAL_THRESHOLD (F1) is reported above for reference ONLY. The operating point
# shipped in the bundle is GATE_THRESHOLD (config) — a high-confidence veto cutoff.
# The F1-optimal point maximises recall at the cost of precision (~0.28 → ~0.5 prec),
# which is wrong for a gate that should only BLOCK when confident.
print(f"\nGate threshold to ship: {GATE_THRESHOLD}  (F1-optimal was {OPTIMAL_THRESHOLD}, "
      f"kept as threshold_f1_optimal in the bundle)")


In [ ]:
# ── Sealed test evaluation (opened once) ──────────────────────────────────────
from sklearn.calibration import calibration_curve

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= OPTIMAL_THRESHOLD).astype(int)

print("=" * 60)
print(f"SEALED TEST — {best_model_name}  (threshold={OPTIMAL_THRESHOLD})")
print("=" * 60)
print(f"ROC-AUC : {roc_auc_score(y_test, test_proba):.3f}")
print(f"PR-AUC  : {average_precision_score(y_test, test_proba):.3f}")
print(f"Precision @ threshold : {precision_score(y_test, test_pred, zero_division=0):.3f}")
print(f"Recall    @ threshold : {recall_score(y_test, test_pred, zero_division=0):.3f}")
print(f"F1        @ threshold : {f1_score(y_test, test_pred, zero_division=0):.3f}")
print(f"Signals               : {test_pred.sum()} / {len(test_pred)}")
print(f"Base rate             : {y_test.mean():.3f}")
print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, test_pred))

# ── Calibration (reliability) curve ───────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(y_test, test_proba, n_bins=10)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(mean_pred, frac_pos, "s-", color="steelblue", label="Stage-1 model")
ax.plot([0, 1], [0, 1], "k:", label="perfectly calibrated")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
ax.set_title(f"MERG {PAIR} — calibration (test)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ── Per-event precision (no single event should dominate) ─────────────────────
test_eval = df_test[["time", "event"]].copy()
test_eval["y"]     = y_test.values
test_eval["proba"] = test_proba
test_eval["pred"]  = test_pred

per_event = (
    test_eval[test_eval["pred"] == 1]
    .groupby("event")
    .agg(n=("y", "size"), correct=("y", "sum"))
)
per_event["precision"] = per_event["correct"] / per_event["n"]
per_event = per_event.sort_values("n", ascending=False)

print("\nPer-event precision among *signals* (pred=1), top 20 by count:")
print(per_event.head(20).round(3).to_string())

# ── Confidence-threshold sweep (sealed test) ──────────────────────────────────
# The F1-optimal threshold (0.26) fires on ~50% of events at only 0.45 precision —
# a gate wants the high-confidence regime instead. Sweep cutoffs to see the real
# operating trade-off on the SEALED test set.
print("\n=== Test-set operating points (confidence sweep) ===")
print(f"{'threshold':>10}{'precision':>11}{'recall':>9}{'F1':>8}{'signals':>9}")
print("-" * 50)
for thr in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    pred = (test_proba >= thr).astype(int)
    n_sig = int(pred.sum())
    if n_sig == 0:
        print(f"{thr:>10.2f}{'—':>11}{'—':>9}{'—':>8}{n_sig:>9}")
        continue
    p = precision_score(y_test, pred, zero_division=0)
    r = recall_score(y_test, pred, zero_division=0)
    f = f1_score(y_test, pred, zero_division=0)
    print(f"{thr:>10.2f}{p:>11.3f}{r:>9.3f}{f:>8.3f}{n_sig:>9}")

# ── Speech vs non-speech breakdown (sealed test) ──────────────────────────────
# Speeches ("... speech", "... testifies") are the known weak spot: far lower base
# rate and the model over-predicts reactions on them. Report the split at BOTH the
# F1-optimal threshold and the intended gate threshold (0.60).
test_eval["is_speech"] = df_test["is_speech"].values

for thr, tag in [(OPTIMAL_THRESHOLD, "F1-optimal"), (GATE_THRESHOLD, "gate")]:
    print(f"\n=== Speech vs non-speech (threshold={thr:.4f} — {tag}) ===")
    for grp, sub in test_eval.groupby("is_speech"):
        label  = "SPEECH" if grp else "NON-SPEECH"
        y_sub  = sub["y"].values
        p_sub  = sub["proba"].values
        pred_sub = (p_sub >= thr).astype(int)
        n_sig = int(pred_sub.sum())
        prec  = precision_score(y_sub, pred_sub, zero_division=0)
        rec   = recall_score(y_sub, pred_sub, zero_division=0)
        roc   = roc_auc_score(y_sub, p_sub)
        print(f"  {label:<12} n={len(sub):>4}  base_rate={y_sub.mean():.3f}  "
              f"ROC-AUC={roc:.3f}  signals={n_sig:>4}  precision={prec:.3f}  recall={rec:.3f}")


In [ ]:
import joblib

# ── Save the trained model + everything it needs at inference time ────────────
# The bundle records the window prefix, feature list, event encoding and threshold so
# the runtime responder (frival/agents/macro_event_responder.py) can reproduce the
# exact feature vector.
model_path = MODELS_DIR / f"{PAIR}_MERG_v2_stage1_{best_model_name}.joblib"
joblib.dump(
    {
        "model": final_model,
        "features": selected_features,
        "threshold": GATE_THRESHOLD,               # gate operating point (runtime default)
        "threshold_f1_optimal": OPTIMAL_THRESHOLD, # F1-optimal, kept for reference only
        "window_prefix": WINDOW_PREFIX,
        "event_top_k": top_k_events,
        "event_min_count": EVENT_MIN_COUNT,
        "reaction_classes": list(REACTION_CLASSES),
        "train_end": TRAIN_END,
        "val_end": VAL_END,
    },
    model_path,
)

print(f"✓ Saved model bundle → {model_path}")
print(f"  model         : {best_model_name}")
print(f"  features      : {len(selected_features)}")
print(f"  threshold     : {GATE_THRESHOLD}  (gate default)")
print(f"  threshold_f1  : {OPTIMAL_THRESHOLD}  (reference)")
print(f"  window_prefix : {WINDOW_PREFIX}")


## Next Steps

This notebook produced **Stage 1 (reaction detector)**. The remaining exploration from the MERG
task definition is:

1. **Incompleto ladder** — re-run with `WINDOW_PREFIX ∈ {5, 6, 7, 10, 15}` and plot
   test ROC-AUC vs prefix. The curve should be monotonic; `15` should reproduce the inflated
   `MERG_v1` (~0.97), which is the leakage baseline. The **leakage verdict**: does the leak-free
   prefix (`5` or `6`) still clear ROC-AUC ≥ 0.58?
2. **Stage 2 (direction)** — a binary U-vs-D classifier trained only on the `y_reaction == 1`
   rows, using the same pipeline.
3. **Event granularity A/B/C** — compare (A) no event feature vs (B) out-of-fold target-encoded
   event vs (C) the one-hot used here, to quantify how much the event name adds.
4. **Time-zone confirmation** — resolve the `time` column provenance (P0) before any live gate.
